# Self-paced reading datasets

In moving-window self-paced reading, a button press reveals the next region and
hides the previous one; intervals between presses provide region reading times.
The region is often one word, but some sources release multiword summaries.
The inventory below is generated from the installed catalog, so it follows
changes to registered datasets.

## Prerequisites

```bash
pip install -e ".[models,frequency,r]"
replica fetch natural_stories
```

The example uses GPT-2, a word-frequency table and a mixed model requiring R with
`lme4`. See [installation](../tutorials/getting_started.ipynb).

In [ ]:
import numpy as np

## 1. Explore all available datasets

In [ ]:
import pandas as pd

from replica.dataset import available_datasets, describe_dataset, load_dataset

# Read the installed catalog without downloading or loading dataset files.
descriptions = [describe_dataset(name) for name in available_datasets()]
inventory = pd.DataFrame(
    [
        {"Dataset": d.name, "Loader": d.loader, "Measures": ", ".join(d.measures)}
        for d in descriptions
        if d.paradigm == "self-paced-reading"
    ]
)
inventory

## 2. Inspect a dataset

`ds.data()` returns `Datum` records whose values align with each record's
annotation layer. `ds.feature(name)` selects a stored quantity for use as a
target, predictor or control; resolution produces aligned `FeatureValues`.

In [ ]:
DATASET_NAME = "natural_stories"
ds = load_dataset(DATASET_NAME)
stimuli = ds.stimuli()
print(f"Dataset:      {ds.info.name}")
print(f"Stimuli:      {len(stimuli)}")
print(f"Participants: {len(ds.participants)}")
print(f"Measures:     {ds.measures}")
print(f"\nFirst stimulus ({len(stimuli[0].layers['words'])} source units):")
print(stimuli[0].text[:200] + "...")

In [ ]:
# Look at observations
data = ds.data(measure=ds.default_measure)
print(f"{len(data)} Datum objects for '{ds.default_measure}'\n")

datum = data[0]
words = datum.stimulus.unit_strings(datum.layer)
print(f"Participant: {datum.participant_id}")
print(f"Stimulus:    {datum.stimulus.stimulus_id}")
print("\nWord-by-word RT (first 10):")
for w, v in list(zip(words, datum.values, strict=True))[:10]:
    val_str = f"{v:7.1f} ms" if not np.isnan(v) else "    NaN"
    print(f"  {w:15s} -> {val_str}")

# RT distribution
all_values = np.concatenate([d.values for d in data])
observed = all_values[np.isfinite(all_values)]
print(f"\nRT distribution ({ds.default_measure}):")
print(f"  Finite observations: {len(observed):,}")
print(
    f"  Mean: {np.mean(observed):.1f} ms  Median: {np.median(observed):.1f} ms  Std: {np.std(observed):.1f} ms"
)

## 3. Evaluate a language model

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

In [ ]:
from replica.analysis import AnalysisConfig, EvaluationSpec, evaluate
from replica.feature import (
    HuggingFaceLM,
    ModelFeature,
    Surprisal,
    UnigramSurprisal,
    UnitLength,
    WordfreqSource,
)

spec = EvaluationSpec(
    targets=[ds.feature(ds.default_measure)],
    predictors=[ModelFeature(HuggingFaceLM("gpt2"), Surprisal())],
    controls=[UnitLength(), UnigramSurprisal(WordfreqSource(ds.info.language))],
)

results = evaluate(
    dataset=ds,
    spec=spec,
    analysis=AnalysisConfig(n_folds=5),  # Up to five folds per holdout axis.
)

for r in results:
    print(f"{r.feature_name} -> {r.dv_name}")
    print(f"  delta-LL:          {r.delta_ll:+.2f}")
    print(f"  delta-LL per unit: {r.delta_ll_per_unit:+.6f}")
    print(f"  n observations:    {r.n_observations}")

### UCL SPR vs UCL ET

UCL SPR and UCL ET share source materials but differ in coverage and stimulus spelling. Each loader retains the text of its own release, including the SPR release's documented typos. For a comparison on matched items, establish the item mapping and use each paradigm's actual text rather than assuming identical unit positions.

```python
from replica.dataset import load_dataset
from replica import EvaluationSpec, HuggingFaceLM, ModelFeature, Surprisal, UnitLength
from replica.analysis import benchmark

spr = load_dataset("ucl_spr")
et = load_dataset("ucl_et")
model = HuggingFaceLM("gpt2")


def spec_for(dataset):
    return EvaluationSpec(
        targets=[dataset.feature(dataset.default_measure)],
        predictors=[ModelFeature(model, Surprisal())],
        controls=[UnitLength()],
    )

results = benchmark([spr, et], [spec_for])
```